# EDA 01 · Analiza Zbioru Respiratory Sound Database ICBHI 2017

<div style="background-color: #f8f9fa; border-left: 4px solid #2b6cb0; padding: 12px 16px; margin-bottom: 20px; border-radius: 0 4px 4px 0;">
  <strong>Praca Inżynierska:</strong> -- <br>
  <strong>Autorzy:</strong> Norbert Gwiazda, Mateusz Ciurzyński | <strong>Data:</strong> 2026-09-02<br>
  <strong>Zbiór danych:</strong> Respiratory Sound Database (ICBHI 2017)<br>
  <strong>Cel:</strong> --
</div>

---

## Spis treści
- [1. Kontekst i Charakterystyka Bazy Danych](#1-kontekst-i-charakterystyka-bazy-danych)
- [2. Środowisko i Import Bibliotek](#2-środowisko-i-import-bibliotek)
- [3. Wczytanie danych](#3-wczytanie-danych)
---

## 1. Kontekst i Charakterystyka Bazy Danych

Zbiór *Respiratory Sound Database* zawiera próbki cykli oddechowych, zebranych niezależnie przez dwa zespoły naukowe z Portugalii i Grecji na przestrzeni kilku: 
- **Zespół portugalski:** *School of Health Sciences* na Uniwersytecie w Aveiro (ESSUA) - nagrania w Laboratorium Badań i Rehabilitacji Oddechowej (**Lab3R**) oraz w Szpitalu Infante D. Pedro w Aveiro.
- **Zespół grecki:** *Arystotelesowski Uniwersytet w Tesalonikach* (AUTH) oraz *Uniwersytet w Coimbrze* (UC) — nagrania w Szpitalu Głównym Papanikolaou w Tesalonikach oraz w Szpitalu Ogólnym w Imathia.

Baza danych zawiera ponad $5,5$ godzin nagrań, zawierających około $6898$ cykli oddechowych. Dane zebrano od $126$ pacjentów. Nazwy plików z nagraniami zawierają metadane określające indeks pacjenta, miejsce badania, sprzęt użyty do badania oraz sposób badania. Dodatkowo zbiór zawiera pliki tekstowe odpowiadające nagraniom określające czy i w którym momencie nagrania występują świsty lub trzaski.

Lokalizacje osłuchu:
- Trachea (Tc) - Tchawica
- Anterior left (Al) - Przód lewy (przednia lewa strona klatki piersiowej)
- Anterior right (Ar) - Przód prawy (przednia prawa strona klatki piersiowej)
- Posterior left (Pl) - Tył lewy (tylna lewa strona klatki piersiowej)
- Posterior right (Pr) - Tył prawy (tylna prawa strona klatki piersiowej)
- Lateral left (Ll) - Bok lewy (boczna lewa strona klatki piersiowej)
- Lateral right (Lr) - Bok prawy (boczna prawa strona klatki piersiowej)

Sprzęt użyty do badania:
- AKG C417L Microphone (AKGC417L), 
- 3M Littmann Classic II SE Stethoscope (LittC2SE), 
- 3M Litmmann 3200 Electronic Stethoscope (Litt3200), 
- WelchAllyn Meditron Master Elite Electronic Stethoscope (Meditron)

Dostępny jest również plik `ICBHI_Challenge_diagnosis.txt` zawierający diagnozę choroby dla każdego pacjenta. Możliwe diagnozy to:
- Zdrowy
- Astma
- Przewlekła Obturacyjna Choroba Płuc (COPD)
- Zakażenie górnych dróg oddechowych (URTI)
- Infekcja dolnych dróg oddechowych (LRTI)
- Rozstrzenie oskrzeli (Bronchiectasis)
- Pneumonia
- Zapalenie oskrzelików (Bronchiolitis)


## 2. Środowisko i Import Bibliotek

In [5]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%config InlineBackend.close_figures = True

# Wyciszenie zbędnych ostrzeżeń
import os
import warnings

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

from pathlib import Path

# Wizualizacja danych
import matplotlib.pyplot as plt

# Przetwarzanie i analiza numeryczna
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

plt.rcParams.update({
    # Fonty i czytelność
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlepad": 12,
    "axes.labelsize": 10,
    "axes.labelweight": "medium",

    # Krawędzie i czyszczenie szumu wizualnego
    "axes.spines.top": False,     # Ukrywa górną ramkę
    "axes.spines.right": False,   # Ukrywa prawą ramkę
    "axes.grid": True,
    "grid.alpha": 0.3,            # Subtelna, jasna siatka pomocnicza
    "grid.linestyle": "--",

    # Dopasowanie rozmiarów
    "figure.figsize": (9, 4.5),
    "figure.autolayout": True,    # Zapobiega obcinaniu podpisów osi
})

## 3. Wczytanie danych

W pierwszej kolejności przyjrzymy się zawartości folderu zbioru danych

In [6]:
DATA_PATH = Path('../data/ICBHI_final_database/')
DIAGNOSIS_FILE = DATA_PATH / 'ICBHI_Challenge_diagnosis.txt'

audio_files = list(DATA_PATH.rglob("*.wav"))

print(f"Wszystkich nagrań: {len(audio_files)}")
print("Pierwsze 5 nagrań:")
for f in audio_files[:5]:
    print(f"  - {f.relative_to(DATA_PATH)}")

print("\n\nZawartość pliku metadanych:")
print(*DIAGNOSIS_FILE.read_text().splitlines()[:5], sep="\n")


Wszystkich nagrań: 920
Pierwsze 5 nagrań:
  - 163_2b2_Ll_mc_AKGC417L.wav
  - 139_1b1_Pr_sc_Litt3200.wav
  - 157_1b1_Al_sc_Meditron.wav
  - 138_2p2_Lr_mc_AKGC417L.wav
  - 177_2b4_Tc_mc_AKGC417L.wav


Zawartość pliku metadanych:
101	URTI
102	Healthy
103	Asthma
104	COPD
105	URTI


Przetworzenie plików audio może być problematyczne, dlatego załadujemy metadane do ramki pandas, a do samych plików audio umieścimy wskaźnik.

In [7]:
annotations = []
file_metadata = []

for file in DATA_PATH.iterdir():
    parts = file.stem.split("_")
    if len(parts) != 5:
        continue

    if file.suffix == ".wav":
        file_metadata.append({"patient_id": int(parts[0]),
                              "recording_id": parts[1],
                              "chest_location": parts[2],
                              "acquisition_mode": parts[3],
                              "recording_equipment": parts[4],
                              "filename": file.stem})
    elif file.suffix == ".txt":
        try:
            anno_df = pd.read_csv(
                file, sep="\t", names=["start", "end", "crackles", "wheezes"]
            )
            anno_df["filename"] = file.stem
            if anno_df is not None:
                annotations.append(anno_df)
        except Exception as e:
            print(f"Pominięto uszkodzony plik adnotacji {file.name}: {e}")

df = pd.concat(annotations, ignore_index=True)
metadata_df = pd.DataFrame(file_metadata)
patients_df = pd.read_csv(
    DIAGNOSIS_FILE, sep="\t", names=["patient_id", "diagnosis"]
)
merged_metadata = pd.merge(
    metadata_df, patients_df, on="patient_id", how="left"
)
df = pd.merge(df, merged_metadata, on="filename", how="left")

c = df.get("crackles", 0) == 1
w = df.get("wheezes", 0) == 1

conditions = [c & w, c, w]
choices = ["Both", "Crackles", "Wheezes"]

df["anomaly_type"] = np.select(conditions, choices, default="Normal")
df = df.drop(columns=["crackles", "wheezes"], errors="ignore")
df = df.reset_index(drop=True)

df.head()

,start,end,filename,patient_id,recording_id,chest_location,acquisition_mode,recording_equipment,diagnosis,anomaly_type
0,1.144,3.848,172_1b3_Al_mc_AKGC417L,172,1b3,Al,mc,AKGC417L,COPD,Normal
1,3.848,6.786,172_1b3_Al_mc_AKGC417L,172,1b3,Al,mc,AKGC417L,COPD,Normal
2,6.786,9.240,172_1b3_Al_mc_AKGC417L,172,1b3,Al,mc,AKGC417L,COPD,Normal
3,9.240,11.962,172_1b3_Al_mc_AKGC417L,172,1b3,Al,mc,AKGC417L,COPD,Normal
4,11.962,14.716,172_1b3_Al_mc_AKGC417L,172,1b3,Al,mc,AKGC417L,COPD,Normal
